# Preparació dels jobs d'AlphaFold3 — IL-7Rα

Aquest notebook genera els inputs i els scripts d'enviament al clúster MareNostrum 5 (MN5) per a les prediccions estructurals amb AlphaFold3 dels 95 complexos target–binder del dataset IL-7Rα.

El fitxer d'entrada és un FASTA on cada entrada conté la seqüència del target (IL-7Rα, 459 aminoàcids, extreta de UniProt) i la del binder corresponent separades per dos punts. A partir d'aquest fitxer, la funció readFastaFile del mòdul alignment de la llibreria prepare_proteins carrega les seqüències i les organitza en un diccionari de cadenes (A = target, B = binder) per a cada complex. La classe sequenceModels valida que tots els aminoàcids siguin estàndards i que el format sigui correcte per a les funcions posteriors.

La funció setUpAlphaFold3 crea l'estructura de carpetes del projecte (af3_jobs_il7ra), genera els fitxers d'entrada en format JSON per a cada complex i escriu les comandes d'execució corresponents. Les comandes no s'executen directament des d'aquí, sinó que es passen a mn5.jobArrays, que genera el fitxer de submissió SLURM (submit_af3_il7ra.sh) configurat per a la partició acc_bscls del MN5. El job s'envia al clúster amb sbatch submit_af3_il7ra.sh.

In [ ]:
import sys
from pathlib import Path

base = Path.cwd().parent
sys.path.append(str(base))

from prepare_proteins._prepare_sequences import sequenceModels
from prepare_proteins._prepare_sequences import alignment
from bsc_calculations import mn5

fasta_file   = base / "Inputs" / "input_af3_il7ra.fasta"
af3_folder   = base / "Scripts" / "af3_jobs_il7ra"
slurm_script = base / "Scripts" / "submit_af3_il7ra.sh"

skip_finished = True
model_seeds   = [1]

raw_sequences = alignment.readFastaFile(str(fasta_file), replace_slash=True)

parsed_sequences = {}
for model_name, seq in raw_sequences.items():
    chains = [s.strip() for s in seq.split(":")]
    parsed_sequences[model_name] = {
        "A": chains[0],
        "B": chains[1],
    }

sequence_models = sequenceModels(parsed_sequences)

af3_jobs = sequence_models.setUpAlphaFold3(
    str(af3_folder),
    model_seeds=model_seeds,
    skip_finished=skip_finished,
)

if af3_jobs:
    mn5.jobArrays(
        af3_jobs,
        script_name=str(slurm_script),
        job_name='af3_il7ra',
        program='alphafold3',
        partition="acc_bscls",
    )

    print(f'Prepared {len(af3_jobs)} AF3 jobs in {af3_folder.resolve()}')
    print(f'Wrote {slurm_script}')
    print(f'Submit with: sbatch {slurm_script}')
    print(af3_jobs[0])